# Lecture 11: GNIS & Baseball Examples

In [1]:
import numpy as np
import pandas as pd

---

# Scalar Functions and Query Plans

In [3]:
## we'll use the Lahman baseball database in our examples today.
%reload_ext sql
%sql postgresql://localhost:5432/baseball
%config SqlMagic.displaylimit = 20

In [4]:
%%sql
WITH year_num AS
  (SELECT year_id, (year_id % 100) as year
     FROM batting
  )
SELECT year_id, CONCAT('''', LPAD(year::text, 2, '0')) as year
  FROM year_num
 LIMIT 5;

Running query in 'postgresql://localhost:5432/baseball'

5 rows affected.

year_id,year
1871,'71
1871,'71
1871,'71
1871,'71
1871,'71


Let's analyze the below query (we've flattened it for convenience):

In [5]:
%%sql
EXPLAIN (VERBOSE true)
SELECT year_id,
       CONCAT('''', LPAD((year_id % 100)::text, 2, '0')) AS year
FROM batting;


Running query in 'postgresql://localhost:5432/baseball'

2 rows affected.

QUERY PLAN
Seq Scan on public.batting (cost=0.00..3927.29 rows=104324 width=36)
"Output: year_id, concat('''', lpad(((year_id % 100))::text, 2, '0'::text))"


What if scalar functions mention multiple tables?

The below query computes an arbitrary statistic for pitchers:
* 1 point for every strikeout they throw as pitcher
* –1 for every point they themselves struck out as batter

If the notebook-like output is hard to read, try out the query in `psql`. Note that notebooks don't preserve whitespace when displaying dataframes.

In [6]:
%%sql
EXPLAIN (VERBOSE true)
SELECT p.player_id, p.so - b.so
  FROM pitching p
  INNER JOIN batting b
  ON p.player_id=b.player_id;

Running query in 'postgresql://localhost:5432/baseball'

11 rows affected.

QUERY PLAN
Nested Loop (cost=0.43..12951.05 rows=339712 width=13)
"Output: p.player_id, (p.so - b.so)"
-> Seq Scan on public.pitching p (cost=0.00..1374.06 rows=45806 width=13)
"Output: p.player_id, p.year_id, p.stint, p.team_id, p.lg_id, p.w, p.l, p.g, p.gs, p.cg, p.sho, p.sv, p.ipouts, p.h, p.er, p.hr, p.bb, p.so, p.baopp, p.era, p.ibb, p.wp, p.hbp, p.bk, p.bfp, p.gf, p.r, p.sh, p.sf, p.gidp"
-> Memoize (cost=0.43..0.73 rows=7 width=13)
"Output: b.so, b.player_id"
Cache Key: p.player_id
Cache Mode: logical
-> Index Scan using batting_pkey on public.batting b (cost=0.42..0.72 rows=7 width=13)
"Output: b.so, b.player_id"


### Window Functions

In [7]:
%%sql
SELECT name_first, name_last, year_id, HR,
       rank() OVER (ORDER BY HR DESC),
       avg(HR)    OVER (PARTITION BY b.player_id ORDER BY year_id ROWS 3 PRECEDING) as avg_3yr,
       lag(HR, 7) OVER (PARTITION BY b.player_id ORDER BY year_id) as previous,
       lag(HR, 2) OVER (PARTITION BY b.player_id ORDER BY year_id) as lag2
FROM batting b, people p
WHERE p.player_id = b.player_id
   AND (name_last = 'Bonds' or name_last = 'Ruth')
ORDER BY HR DESC
LIMIT 10;

Running query in 'postgresql://localhost:5432/baseball'

10 rows affected.

name_first,name_last,year_id,hr,rank,avg_3yr,previous,lag2
Barry,Bonds,2001,73,1,48.2500000000000000,37,34
Babe,Ruth,1927,60,2,44.5000000000000000,54,25
Babe,Ruth,1921,59,3,38.2500000000000000,0,29
Babe,Ruth,1920,54,4,24.0000000000000000,None,11
Babe,Ruth,1928,54,4,46.5000000000000000,59,47
Barry,Bonds,2000,49,6,40.0000000000000000,46,37
Babe,Ruth,1930,49,6,52.2500000000000000,41,54
Babe,Ruth,1926,47,8,39.7500000000000000,29,46
Barry,Bonds,1993,46,9,34.5000000000000000,16,25
Barry,Bonds,2002,46,9,50.5000000000000000,33,49


Same query, different order by - so that we can inspect the other attributes

In [11]:
%%sql
SELECT name_first, name_last, year_id, HR,
       rank() OVER (ORDER BY HR DESC),
       avg(HR)    OVER (PARTITION BY b.player_id ORDER BY year_id ROWS 3 PRECEDING) as avg_3yr,
       lag(HR, 7) OVER (PARTITION BY b.player_id ORDER BY year_id) as previous,
       lag(HR, 2) OVER (PARTITION BY b.player_id ORDER BY year_id) as lag2
FROM batting b, people p
WHERE p.player_id = b.player_id
   AND (name_last = 'Bonds' or name_last = 'Ruth')
ORDER BY b.player_id, year_id 
LIMIT 20;

Running query in 'postgresql://localhost:5432/baseball'

20 rows affected.

name_first,name_last,year_id,hr,rank,avg_3yr,previous,lag2
Barry,Bonds,1986,16,47,16.0000000000000000,None,None
Barry,Bonds,1987,25,39,20.5000000000000000,None,None
Barry,Bonds,1988,24,43,21.6666666666666667,None,16
Barry,Bonds,1989,19,46,21.0000000000000000,None,25
Barry,Bonds,1990,33,28,25.2500000000000000,None,24
Barry,Bonds,1991,25,39,25.2500000000000000,None,19
Barry,Bonds,1992,34,25,27.7500000000000000,None,33
Barry,Bonds,1993,46,9,34.5000000000000000,16,25
Barry,Bonds,1994,37,21,35.5000000000000000,25,34
Barry,Bonds,1995,33,28,37.5000000000000000,24,46


### Inverse Distribution Window Functions

In [12]:
%%sql
SELECT MIN(HR),
       percentile_cont(0.25) WITHIN GROUP (ORDER BY HR) AS p25,
       percentile_cont(0.50) WITHIN GROUP (ORDER BY HR) AS median,
       percentile_cont(0.75) WITHIN GROUP (ORDER BY HR) AS p75,
       percentile_cont(0.99) WITHIN GROUP (ORDER BY HR) AS p99,
       MAX(HR),
       AVG(HR) AS "average hit rate"
FROM batting;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

min,p25,median,p75,p99,max,average hit rate
0,0.0,0.0,2.0,31.0,73,2.8315823779763046


In [13]:
%%sql
SELECT HR, COUNT(*) FROM batting GROUP BY HR ORDER BY HR DESC;

Running query in 'postgresql://localhost:5432/baseball'

67 rows affected.

hr,count
73,1
70,1
66,1
65,1
64,1
63,1
61,1
60,1
59,2
58,3


### Hypothetical-Set Window Functions

In [24]:
hrs = 70 # hypothetically, four home runs

In [25]:
%%sql
SELECT {{hrs}} as hypothetical,
       rank({{hrs}}) WITHIN GROUP (ORDER BY HR DESC),
       dense_rank({{hrs}}) WITHIN GROUP (ORDER BY HR DESC),
       percent_rank({{hrs}}) WITHIN GROUP (ORDER BY HR DESC) * 100 AS pct_rank,
       cume_dist({{hrs}}) WITHIN GROUP (ORDER BY HR)
FROM batting
LIMIT 10;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

hypothetical,rank,dense_rank,pct_rank,cume_dist
70,2,2,0.000958552202752962,0.9999904145698538


Without `jupysql` variable substituion

In [19]:
%%sql
SELECT 4 as hypothetical,
       rank(4) WITHIN GROUP (ORDER BY HR DESC),
       dense_rank(4) WITHIN GROUP (ORDER BY HR DESC),
       percent_rank(4) WITHIN GROUP (ORDER BY HR DESC) * 100 AS pct_rank,
       cume_dist(4) WITHIN GROUP (ORDER BY HR)
FROM batting
LIMIT 10;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

hypothetical,rank,dense_rank,pct_rank,cume_dist
4,18420,63,17.655573022506807,0.823445962137551


# GNIS

This notebook transforms the existing [Geographics Names Information Systems (GNIS)](https://www.usgs.gov/core-science-systems/ngp/board-on-geographic-names/download-gnis-data) national zip file.

We have provided a subset of the sql database for you in `data/national.sql`.

If you'd like to make your own version of this database, see the end of this notebook. Note: Because of its size, we don't recommend building the GNIS SQL database from scratch on DataHub.


In [26]:
!psql -h localhost -d gnis -c 'SELECT pg_terminate_backend(pg_stat_activity.pid) FROM pg_stat_activity WHERE datname = current_database() AND pid <> pg_backend_pid();'
!psql -h localhost -c 'DROP DATABASE IF EXISTS gnis'
!psql -h localhost -c 'CREATE DATABASE gnis' 
!psql -h localhost -d gnis -f data/gnis.sql

 pg_terminate_backend 
----------------------
(0 rows)

DROP DATABASE
CREATE DATABASE
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
CREATE TABLE
ALTER TABLE
CREATE TABLE
ALTER TABLE
COPY 3195
COPY 11533
CREATE INDEX


In [27]:
%reload_ext sql
%sql postgresql://localhost:5432/gnis
%config SqlMagic.displaylimit = 15

Connecting and switching to connection 'postgresql://localhost:5432/gnis'

* View schema in `psql`
* View some rows below

In [28]:
%sql SELECT COUNT(*) FROM national;

Running query in 'postgresql://localhost:5432/gnis'

1 rows affected.

count
11533


In [29]:
%sql SELECT * FROM national WHERE county_name = 'Alameda';

Running query in 'postgresql://localhost:5432/gnis'

21 rows affected.

feature_id,feature_name,feature_class,state_alpha,state_numeric,county_name,county_numeric,primary_lat_dms,prim_long_dms,prim_lat_dec,prim_long_dec,source_lat_dms,source_long_dms,source_lat_dec,source_long_dec,elev_in_m,elev_in_ft,map_name,date_created,date_edited
218316,Apperson Creek,Stream,CA,6,Alameda,1.0,373349N,1215000W,37.5635453,-121.8332887,373232N,1214804W,37.5422222,-121.8011111,148.0,486.0,La Costa Valley,01/19/1981,None
225998,Irvington High School,School,CA,6,Alameda,1.0,373126N,1215801W,37.523814,-121.9670659,None,None,None,None,13.0,43.0,Niles,01/19/1981,03/31/2021
226951,Laurel Elementary School,School,CA,6,Alameda,1.0,374734N,1221147W,37.792899,-122.1964288,None,None,None,None,68.0,223.0,Oakland East,06/14/2000,03/14/2021
229367,Murray Elementary School,School,CA,6,Alameda,1.0,374318N,1215557W,37.721801,-121.9326269,None,None,None,None,112.0,367.0,Dublin,01/19/1981,03/14/2021
235581,Strawberry Creek,Stream,CA,6,Alameda,1.0,375221N,1221443W,37.8724258,-122.2452464,375251N,1221354W,37.8807588,-122.2316349,154.0,505.0,Oakland East,01/19/1981,08/31/2016
1654274,Hayward Golf Course,Locale,CA,6,Alameda,1.0,373726N,1220250W,37.6238222,-122.0471843,None,None,None,None,5.0,16.0,Newark,01/19/1981,None
1664964,KOFY-AM (San Mateo),Tower,CA,6,Alameda,1.0,374934N,1221842W,37.8260385,-122.3116366,None,None,None,None,2.0,7.0,Oakland West,07/01/1994,None
1670278,Lake Elizabeth,Lake,CA,6,Alameda,1.0,373255N,1215742W,37.5487056,-121.9617554,None,None,None,None,16.0,52.0,Niles,11/09/1995,03/07/2019
1692819,California School for the Deaf - Fremont,School,CA,6,Alameda,1.0,373334N,1215747W,37.5593966,-121.9631843,None,None,None,None,20.0,66.0,Niles,05/08/1996,09/16/2016
1692863,J A Freitas Library,Building,CA,6,Alameda,1.0,374335N,1220925W,37.7263185,-122.1569101,None,None,None,None,19.0,62.0,San Leandro,05/08/1996,None


In [30]:
%%sql
SELECT *
FROM national TABLESAMPLE BERNOULLI(10);

Running query in 'postgresql://localhost:5432/gnis'

1171 rows affected.

feature_id,feature_name,feature_class,state_alpha,state_numeric,county_name,county_numeric,primary_lat_dms,prim_long_dms,prim_lat_dec,prim_long_dec,source_lat_dms,source_long_dms,source_lat_dec,source_long_dec,elev_in_m,elev_in_ft,map_name,date_created,date_edited
4855,Fresnal Wash,Stream,AZ,4,Graham,9.0,324247N,1101159W,32.7131243,-110.1998064,324444N,1100603W,32.7456241,-110.100916,1217.0,3993.0,Eureka Ranch,02/08/1980,None
8053,Milkweed Spring,Spring,AZ,4,Mohave,15.0,353706N,1134223W,35.6184467,-113.7063277,None,None,None,None,1441.0,4728.0,Milkweed Canyon SW,02/08/1980,04/18/2011
8388,Mud Tank,Reservoir,AZ,4,Apache,1.0,333627N,1094302W,33.6075041,-109.7173476,None,None,None,None,2125.0,6972.0,Maverick SW,02/08/1980,03/19/2019
20915,Mount Burro,Summit,AZ,4,Coconino,5.0,361000N,1123743W,36.1666491,-112.6285208,None,None,None,None,1734.0,5689.0,Supai,06/27/1984,03/22/2011
21265,Flores Railroad Station,Building,AZ,4,Yavapai,25.0,340356N,1124923W,34.0655834,-112.8229583,None,None,None,None,794.0,2605.0,Flores,06/27/1984,None
21597,Dad Patterson Tank,Reservoir,AZ,4,Apache,1.0,343207N,1094226W,34.5353169,-109.7072364,None,None,None,None,1709.0,5607.0,Hunt,06/27/1984,04/12/2019
22846,Mountain View Elementary School,School,AZ,4,Mohave,15.0,350843N,1143358W,35.1453119,-114.5662344,None,None,None,None,185.0,607.0,Davis Dam,06/27/1984,01/07/2017
26075,Bear Canyon Well,Well,AZ,4,Greenlee,11.0,332032N,1092803W,33.3422811,-109.4675803,None,None,None,None,1518.0,4980.0,Bee Canyon,02/08/1980,02/15/2018
27581,Chilson Camp,Locale,AZ,4,Yavapai,25.0,340504N,1112933W,34.0844817,-111.492638,None,None,None,None,1726.0,5663.0,Mazatzal Peak,02/08/1980,None
27722,Clino Tank,Reservoir,AZ,4,Navajo,17.0,341409N,1101908W,34.2359182,-110.3189522,None,None,None,None,2063.0,6768.0,Limestone Canyon North,02/08/1980,03/20/2019


# Numerical Granularity

In [31]:
%sql SELECT elev_in_m FROM National LIMIT 2;

Running query in 'postgresql://localhost:5432/gnis'

2 rows affected.

elev_in_m
931.0
2707.0


In [32]:
%%sql
SELECT elev_in_m, 
    (elev_in_m / 100)::INTEGER AS quantized,
    ((elev_in_m / 100)::INTEGER) * 100 AS round_to_100,
    SUBSTRING(elev_in_m::TEXT, 1, 2),
    CONCAT(SUBSTRING(elev_in_m::TEXT, 1, 2), '00') AS substring2
FROM National
LIMIT 5;

Running query in 'postgresql://localhost:5432/gnis'

5 rows affected.

elev_in_m,quantized,round_to_100,substring,substring2
931.0,9,900,93,9300
2707.0,27,2700,27,2700
275.0,3,300,27,2700
1685.0,17,1700,16,1600
1354.0,14,1400,13,1300


In [ ]:
%config SqlMagic.named_parameters=True

In [33]:
right_shift = '>>'
left_shift = '<<'

In [34]:
%%sql
/* Since jupysql does not like bitshifts, we can fake it with string interoplation. */
SELECT elev_in_m,
    (16::INTEGER::BIT(12)) AS bit12,
    (16::INTEGER::BIT(12)) {{left_shift}} 3
FROM national
LIMIT 5;

Running query in 'postgresql://localhost:5432/gnis'

5 rows affected.

elev_in_m,bit12,?column?
931.0,000000010000,000010000000
2707.0,000000010000,000010000000
275.0,000000010000,000010000000
1685.0,000000010000,000010000000
1354.0,000000010000,000010000000


In [35]:
%%sql
EXPLAIN (verbose true)
WITH shifts AS (
    SELECT elev_in_m,
       (elev_in_m::integer::bit(12)) AS bit12,
       (elev_in_m::integer::bit(12) {{right_shift}} 8) AS rightshifted,
       ((elev_in_m::integer::bit(12) {{right_shift}} 8) {{left_shift}} 8)::integer AS round_to_256,
       ((elev_in_m::integer::bit(12) {{right_shift}} 8) {{left_shift}} 8)::integer % 256 AS test
  FROM national
)
SELECT COUNT(DISTINCT elev_in_m) AS elevation_meters_count,
       COUNT(DISTINCT bit12) AS bit12_count,
       COUNT(DISTINCT rightshifted) AS rightshift_count,
       COUNT(DISTINCT round_to_256) AS rounded_count
  FROM shifts;

Running query in 'postgresql://localhost:5432/gnis'

4 rows affected.

QUERY PLAN
Aggregate (cost=799.99..800.00 rows=1 width=32)
"Output: count(DISTINCT ""national"".elev_in_m), count(DISTINCT ((""national"".elev_in_m)::integer)::bit(12)), count(DISTINCT (((""national"".elev_in_m)::integer)::bit(12) >> 8)), count(DISTINCT (((((""national"".elev_in_m)::integer)::bit(12) >> 8) << 8))::integer)"
"-> Seq Scan on public.""national"" (cost=0.00..396.33 rows=11533 width=8)"
"Output: ""national"".feature_id, ""national"".feature_name, ""national"".feature_class, ""national"".state_alpha, ""national"".state_numeric, ""national"".county_name, ""national"".county_numeric, ""national"".primary_lat_dms, ""national"".prim_long_dms, ""national"".prim_lat_dec, ""national"".prim_long_dec, ""national"".source_lat_dms, ""national"".source_long_dms, ""national"".source_lat_dec, ""national"".source_long_dec, ""national"".elev_in_m, ""national"".elev_in_ft, ""national"".map_name, ""national"".date_created, ""national"".date_edited"


# Demo 1: Roll-up / Drill-down Practice

Let's start with county-level data on elevations:

In [36]:
%%sql
SELECT state_numeric, county_numeric,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
FROM national TABLESAMPLE BERNOULLI(10)
GROUP BY state_numeric, county_numeric;

Running query in 'postgresql://localhost:5432/gnis'

855 rows affected.

state_numeric,county_numeric,avg,stddev,count
48,383.0,724.0,None,1
12,113.0,31.0,None,1
56,35.0,2460.5,485.78235867515815,2
1,33.0,146.0,None,1
31,97.0,345.0,None,1
46,47.0,1001.0,None,1
39,69.0,205.0,None,1
29,169.0,241.0,None,1
36,119.0,61.0,None,1
36,65.0,156.0,None,1


**Roll up** to state level.
* We save the view as `state_elevations` for later...

In [37]:
%%sql
DROP VIEW IF EXISTS state_elevations;

CREATE VIEW state_elevations AS (
    SELECT state_numeric,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
    FROM national
    GROUP BY state_numeric
);

Running query in 'postgresql://localhost:5432/gnis'

++
||
++
++

In [38]:
%sql SELECT * FROM state_elevations;

Running query in 'postgresql://localhost:5432/gnis'

59 rows affected.

state_numeric,avg,stddev,count
54,363.6190476190476,199.26650831834746,204
29,246.09152542372883,80.2483078596168,343
68,6.666666666666667,7.99166232186187,14
4,1315.3798076923076,672.6305522946129,208
34,40.08943089430894,59.88896941733248,123
51,254.55197132616487,260.54513270095333,283
70,18.333333333333332,31.75426480542942,3
10,22.11111111111111,28.015563440198648,27
35,1756.8467432950192,471.8002505531821,273
45,122.83240223463687,123.96059930539184,181


**Drill down** to include feature class.

In [39]:
%%sql
SELECT state_numeric, feature_class,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
FROM national TABLESAMPLE Bernoulli(10)
GROUP BY state_numeric, feature_class
ORDER BY count(*) DESC;

Running query in 'postgresql://localhost:5432/gnis'

550 rows affected.

state_numeric,feature_class,avg,stddev,count
30,Well,1082.7142857142858,352.20195879668216,14
6,School,108.9090909090909,135.04551421313818,11
48,Building,312.0,369.91860966674517,10
47,Church,173.4,80.54563923639815,10
39,Populated Place,259.0,42.68196340376108,10
47,Cemetery,215.44444444444446,111.11380552288621,9
48,Locale,493.22222222222223,412.1973974256078,9
6,Park,141.875,202.11201809180685,8
26,School,216.125,40.779678062205726,8
35,Well,1404.25,267.56935015591216,8


# Demo 2: Connections to Statistics

## Roll up with marginal distributions

In [40]:
%%sql
SELECT state_numeric,
       AVG(elev_in_m),
       STDDEV(elev_in_m), COUNT(*),
       SUM(COUNT(*)) OVER () AS total,
       COUNT(*)/SUM(COUNT(*)) OVER () AS marginal
FROM national TABLESAMPLE Bernoulli(.07)
GROUP BY state_numeric;

Running query in 'postgresql://localhost:5432/gnis'

6 rows affected.

state_numeric,avg,stddev,count,total,marginal
6,7.0,None,1,6,0.16666666666666666667
12,3.0,None,1,6,0.16666666666666666667
23,None,None,1,6,0.16666666666666666667
24,45.0,None,1,6,0.16666666666666666667
51,641.0,None,1,6,0.16666666666666666667
55,253.0,None,1,6,0.16666666666666666667


In [41]:
%%sql
SELECT COUNT(DISTINCT county_numeric) FROM national;

Running query in 'postgresql://localhost:5432/gnis'

1 rows affected.

count
291


## Drill down with normally-distributed elevations:

Start with the `state_elevations` view from earlier:

In [42]:
%sql SELECT * FROM state_elevations;

Running query in 'postgresql://localhost:5432/gnis'

59 rows affected.

state_numeric,avg,stddev,count
54,363.6190476190476,199.26650831834746,204
29,246.09152542372883,80.2483078596168,343
68,6.666666666666667,7.99166232186187,14
4,1315.3798076923076,672.6305522946129,208
34,40.08943089430894,59.88896941733248,123
51,254.55197132616487,260.54513270095333,283
70,18.333333333333332,31.75426480542942,3
10,22.11111111111111,28.015563440198648,27
35,1756.8467432950192,471.8002505531821,273
45,122.83240223463687,123.96059930539184,181


The `fips_counties` relation has all counties, including those not in `national`:

In [43]:
%sql SELECT * FROM fips_counties LIMIT 10;

Running query in 'postgresql://localhost:5432/gnis'

10 rows affected.

fips,county,state_numeric
1000,Alabama,1
1001,Autauga County,1
1003,Baldwin County,1
1005,Barbour County,1
1007,Bibb County,1
1009,Blount County,1
1011,Bullock County,1
1013,Butler County,1
1015,Calhoun County,1
1017,Chambers County,1


If we wanted to **drill down** to the FIPS counties, we'd need to simulate an elevation for those counties that don't exist in `national`.

Here's the first step in that process, which creates a simulated value for *every* county in `fips_counties`.
* The value is simulated from a normal distribution using that state's elevation statistics (average, standard deviation).
* Just like a Python package, we would need to import `tablefunc` in order to use the `normal_rand` function.

In [44]:
%sql CREATE EXTENSION IF NOT EXISTS tablefunc;

Running query in 'postgresql://localhost:5432/gnis'

++
||
++
++

In [45]:
%%sql
WITH state_cty AS
(SELECT s.state_numeric, f.fips as county_numeric, s.avg, s.stddev, s.count
  FROM state_elevations s, fips_counties f
  WHERE s.state_numeric = f.state_numeric
)
SELECT s.*,
       n.n AS elev_in_m,
       true as elev_in_m_sim -- user-facing flag
  FROM state_cty s,
       LATERAL normal_rand(CAST(s.count AS INTEGER), s.avg, s.stddev) AS n
LIMIT 10;

Running query in 'postgresql://localhost:5432/gnis'

10 rows affected.

state_numeric,county_numeric,avg,stddev,count,elev_in_m,elev_in_m_sim
1,1000,146.37888198757764,102.92185851771194,339,286.68835826438976,True
1,1000,146.37888198757764,102.92185851771194,339,103.79659423552548,True
1,1000,146.37888198757764,102.92185851771194,339,168.8474474318065,True
1,1000,146.37888198757764,102.92185851771194,339,130.48618620283284,True
1,1000,146.37888198757764,102.92185851771194,339,301.0227700361809,True
1,1000,146.37888198757764,102.92185851771194,339,236.1775541823482,True
1,1000,146.37888198757764,102.92185851771194,339,5.605233823243594,True
1,1000,146.37888198757764,102.92185851771194,339,210.34158359856548,True
1,1000,146.37888198757764,102.92185851771194,339,65.76928545625617,True
1,1000,146.37888198757764,102.92185851771194,339,327.277996313111,True


# Assembling an Explicit Hierarchy

In [46]:
## we'll use the Lahman baseball database in our initial examples today.
## replace the database connection with a database of your own!
%reload_ext sql
%sql postgresql://localhost:5432/baseball

Switching to connection 'postgresql://localhost:5432/baseball'

Two relations have the pieces of the hierarchy we want:

In [47]:
%sql SELECT * FROM Appearances WHERE year_id > 1970 LIMIT 2;

Running query in 'postgresql://localhost:5432/baseball'

2 rows affected.

year_id,team_id,lg_id,player_id,g_all,gs,g_batting,g_defense,g_p,g_c,g_1b,g_2b,g_3b,g_ss,g_lf,g_cf,g_rf,g_of,g_dh,g_ph,g_pr
1971,ATL,NL,aaronha01,139,129,139,129,0,0,71,0,0,0,0,0,60,60,0,10,0
1971,ATL,NL,aaronto01,25,10,25,18,0,0,11,0,7,0,0,0,0,0,0,8,0


In [48]:
%sql SELECT * FROM Teams LIMIT 1;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

year_id,lg_id,team_id,franch_id,div_id,rank,g,ghome,w,l,divwin,wcwin,lgwin,wswin,r,ab,h,h2b,h3b,hr,bb,so,sb,cs,hbp,sf,ra,er,era,cg,sho,sv,ipouts,ha,hra,bba,soa,e,dp,fp,name,park,attendance,bpf,ppf,team_idbr,team_idlahman45,team_idretro
1871,NA,BS1,BNA,None,3,31,None,20,10,None,None,N,None,401,1372,426,70,37,3,60,19,73,16,None,None,303,109,3.55,22,1,3,828,367,2,42,23,243,24,0.834,Boston Red Stockings,South End Grounds I,None,103,98,BOS,BS1,BS1


Let's join these two to make our hierarchy! Which way should we make this?

In [49]:
%%sql
SELECT a.player_id, a.team_id, t.div_id, a.*
FROM Appearances a
NATURAL JOIN Teams t
WHERE a.year_id = 2015
LIMIT 100;

Running query in 'postgresql://localhost:5432/baseball'

100 rows affected.

player_id,team_id,div_id,year_id,team_id_1,lg_id,player_id_1,g_all,gs,g_batting,g_defense,g_p,g_c,g_1b,g_2b,g_3b,g_ss,g_lf,g_cf,g_rf,g_of,g_dh,g_ph,g_pr
alvarda02,BAL,E,2015,BAL,AL,alvarda02,12,10,12,12,0,0,0,0,0,0,0,1,12,12,0,0,0
brachbr01,BAL,E,2015,BAL,AL,brachbr01,62,0,5,62,62,0,0,0,0,0,0,0,0,0,0,0,0
brittza01,BAL,E,2015,BAL,AL,brittza01,64,0,2,64,64,0,0,0,0,0,0,0,0,0,0,0,0
cabrace01,BAL,E,2015,BAL,AL,cabrace01,2,0,0,2,2,0,0,0,0,0,0,0,0,0,0,0,0
cabreev01,BAL,E,2015,BAL,AL,cabreev01,29,28,29,28,0,0,0,2,0,27,0,0,0,0,0,0,1
chenwe02,BAL,E,2015,BAL,AL,chenwe02,31,31,0,31,31,0,0,0,0,0,0,0,0,0,0,0,0
clevest01,BAL,E,2015,BAL,AL,clevest01,30,24,30,10,0,9,1,0,0,0,0,0,0,0,18,4,0
davisch02,BAL,E,2015,BAL,AL,davisch02,160,159,160,138,0,0,111,0,0,0,0,0,30,30,22,0,0
deazaal01,BAL,E,2015,BAL,AL,deazaal01,30,27,30,27,0,0,0,0,0,0,19,0,13,27,0,3,0
drakeol01,BAL,E,2015,BAL,AL,drakeol01,13,0,1,13,13,0,0,0,0,0,0,0,0,0,0,0,0


In [50]:
%%sql
CREATE OR REPLACE VIEW bball_tree AS (
    SELECT DISTINCT
        a.player_id, a.team_id, t.div_id,
        a.lg_id, a.year_id
    FROM appearances a
    NATURAL JOIN teams t
);

Running query in 'postgresql://localhost:5432/baseball'

++
||
++
++

In [51]:
%sql SELECT * FROM bball_tree LIMIT 25;

Running query in 'postgresql://localhost:5432/baseball'

25 rows affected.

player_id,team_id,div_id,lg_id,year_id
gumbeha01,NY1,None,NL,1935
gradymi01,SLN,None,NL,1897
deshoji01,WS1,None,AL,1938
prattla01,BRF,None,FL,1915
thompsa01,PHI,None,NL,1890
hollica01,DET,None,AL,1922
halege01,SLA,None,AL,1916
mamaual01,NYA,None,AL,1924
henryji01,BOS,None,AL,1937
cristch01,PHI,None,NL,1906


### Revisiting the Home Run Query

Recall our old home run query:

In [52]:
%%sql
SELECT name_first, name_last, year_id,
       MIN(hr), MAX(hr), AVG(hr), STDDEV(hr), SUM(hr)
FROM batting b, people p
WHERE b.player_id = p.player_id
GROUP BY name_last, name_first, year_id
ORDER BY max DESC
LIMIT 10;

Running query in 'postgresql://localhost:5432/baseball'

10 rows affected.

name_first,name_last,year_id,min,max,avg,stddev,sum
Barry,Bonds,2001,73,73,73.0000000000000000,None,73
Mark,McGwire,1998,70,70,70.0000000000000000,None,70
Sammy,Sosa,1998,66,66,66.0000000000000000,None,66
Mark,McGwire,1999,65,65,65.0000000000000000,None,65
Sammy,Sosa,2001,64,64,64.0000000000000000,None,64
Sammy,Sosa,1999,63,63,63.0000000000000000,None,63
Roger,Maris,1961,61,61,61.0000000000000000,None,61
Babe,Ruth,1927,60,60,60.0000000000000000,None,60
Babe,Ruth,1921,59,59,59.0000000000000000,None,59
Giancarlo,Stanton,2017,59,59,59.0000000000000000,None,59


Set up for roll up/drill down on `bball_tree` hierarchy.
* Join each (raw) person with the associated `bball_tree` entry by `(playerid, yearid)` in a CTE
* Use this result for roll-up and drill-down.

(blank space before we get to the next exercise....)
<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>

In [53]:
%%sql
WITH batting_tree AS (
    SELECT b.*, t.div_id
    FROM batting b, bball_tree t
    WHERE b.player_id = t.player_id
      AND b.year_id = t.year_id
)
SELECT name_first, name_last,
       bt.team_id, bt.lg_id, bt.div_id, bt.year_id,
       MIN(hr), MAX(hr), AVG(hr), STDDEV(hr), SUM(hr)
FROM batting_tree bt, people p
WHERE bt.player_id = p.player_id
GROUP BY bt.player_id, bt.team_id, bt.lg_id, bt.div_id, bt.year_id, name_last, name_first
ORDER BY max DESC
LIMIT 10;


Running query in 'postgresql://localhost:5432/baseball'

10 rows affected.

name_first,name_last,team_id,lg_id,div_id,year_id,min,max,avg,stddev,sum
Barry,Bonds,SFN,NL,W,2001,73,73,73.0000000000000000,None,73
Mark,McGwire,SLN,NL,C,1998,70,70,70.0000000000000000,None,70
Sammy,Sosa,CHN,NL,C,1998,66,66,66.0000000000000000,None,66
Mark,McGwire,SLN,NL,C,1999,65,65,65.0000000000000000,None,65
Sammy,Sosa,CHN,NL,C,2001,64,64,64.0000000000000000,None,64
Sammy,Sosa,CHN,NL,C,1999,63,63,63.0000000000000000,None,63
Roger,Maris,NYA,AL,None,1961,61,61,61.0000000000000000,None,61
Babe,Ruth,NYA,AL,None,1927,60,60,60.0000000000000000,None,60
Babe,Ruth,NYA,AL,None,1921,59,59,59.0000000000000000,None,59
Giancarlo,Stanton,MIA,NL,E,2017,59,59,59.0000000000000000,None,59
